# Практика: перебор настроек и сдача эксперимента

Собираем итог модуля: таблица опытов, честная финальная оценка, отчёт сервису.

In [ ]:
from pathlib import Path
import pandas as pd


def find_digits_csv() -> Path:
    for p in (Path('digits.csv'), Path('../../data/digits.csv'), Path('../data/digits.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('digits.csv не найден — положите файл рядом с ноутбуком')


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

import itertools
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X_rest, X_final, y_rest, y_final = train_test_split(
    df[PIXELS], df['label'], test_size=0.2, random_state=0, stratify=df['label'])
X_fit, X_val, y_fit, y_val = train_test_split(
    X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)


## 1. Все пары настроек сразу

`itertools.product` даёт декартово произведение — то самое, что вы считали двойным циклом на паре 24.

Соберите `grid = list(itertools.product(K_VALUES, SCALING))` для `K_VALUES = [1, 3, 5, 7, 9]` и `SCALING = [False, True]`.

In [ ]:
K_VALUES = [1, 3, 5, 7, 9]
SCALING = [False, True]
grid = None
assert grid is not None and len(grid) == 10
assert all(len(c) == 2 for c in grid)
print(grid[:4])

## 2. Прогон по сетке

Для каждой пары `(k, scaled)` обучите kNN на обучающей части и посчитайте точность на **проверочной**. При `scaled=True` считайте min/max по обучающей части.

Соберите `exp_table` со столбцами `k`, `scaled`, `accuracy`.

**Наблюдение:** здесь все пиксели уже в одной шкале 0…16 — посмотрите, меняет ли масштабирование результат.

In [ ]:
exp_table = None
assert exp_table is not None and len(exp_table) == 10
assert list(exp_table.columns) == ['k', 'scaled', 'accuracy']
assert float(exp_table['accuracy'].max()) > 0.9
print(exp_table)

## 3. Подмножества признаков

Постройте четыре признака: `ink`, `n_dark`, `top_ink` (сумма первых 32 пикселей), `bottom_ink` (сумма последних 32).

`itertools.combinations(FEATURES, 2)` — все пары признаков. Для каждой пары обучите kNN (k=5) и соберите `combo_table` со столбцами `features`, `accuracy`.

In [ ]:
FEATURES = ['ink', 'n_dark', 'top_ink', 'bottom_ink']
combo_table = None
assert combo_table is not None and len(combo_table) == 6
assert list(combo_table.columns) == ['features', 'accuracy']
assert float(combo_table['accuracy'].max()) < 0.9
print(combo_table)

## 4. Выбор конфигурации и один выстрел

Возьмите лучшую строку `exp_table` по проверочной точности -> `best_k`, `best_scaled`. Обучите модель с этой настройкой и посчитайте точность на **финальной** части -> `acc_final`.

Baseline самой частой цифры на финальной части -> `baseline_final`.

In [ ]:
best_k = None
best_scaled = None
acc_final = None
baseline_final = None
assert best_k is not None and best_scaled is not None
assert acc_final is not None and float(acc_final) > 0.9
assert baseline_final is not None and float(baseline_final) < 0.2
print(best_k, best_scaled, round(float(acc_final), 4), round(float(baseline_final), 4))

## 5. Что было бы, если выбирать по финальной части

Посчитайте точность **всех** настроек сетки на финальной части и возьмите максимум -> `acc_peek`. Разница `optimism = acc_peek - acc_final`.

В `OPTIMISM_NOTE` объясните, почему выбор по финальной части — это подгонка, даже если разница маленькая.

In [ ]:
acc_peek = None
optimism = None
OPTIMISM_NOTE = ''
assert acc_peek is not None and optimism is not None
assert float(optimism) >= 0
assert len(OPTIMISM_NOTE) > 80
print(round(float(acc_peek), 4), round(float(optimism), 4))

## 6. Таблица и график для отчёта

Сохраните `exp_table` в `experiments.csv` -> `csv_path`. Постройте график: точность на проверочной части по `k` (две линии — с масштабированием и без) и сохраните `figures/config_accuracy.png` -> `plot_path`.

In [ ]:
from pathlib import Path as _P
import matplotlib.pyplot as plt

_P('figures').mkdir(exist_ok=True)
csv_path = None
plot_path = None
assert csv_path is not None and _P(csv_path).exists()
assert plot_path is not None and _P(plot_path).exists()
print(csv_path, plot_path)

## 7. Чек-лист сдачи

Отметьте `True` только то, что действительно сделано (см. [artifact/PROJECT.md](../../artifact/PROJECT.md)). Число взглядов на финальную часть -> `n_final_looks`.

In [ ]:
acceptance = pd.Series(
    [False, False, False, False, False, False],
    index=['baseline', 'protocol', 'experiments_csv', 'figure', 'final_score', 'limitations'],
)
n_final_looks = None
assert bool(acceptance.all())
assert n_final_looks is not None and int(n_final_looks) == 1
print(acceptance)

## 8. Отчёт сервису

`REPORT` (≥250 символов): что за данные, какой протокол, какая настройка выбрана, какая точность против baseline, какие ограничения (8×8, подвыборка, скорость kNN).

`READY = True` — только если чек-лист заполнен и файлы сохранены.

In [ ]:
REPORT = ''
READY = False
assert len(REPORT) > 250
assert 'baseline' in REPORT.lower()
assert READY is True
print(REPORT)

## 9. Расширение: где kNN станет неудобен

В `NEXT_MODULE` (≥80 символов) объясните, что произойдёт со временем ответа, если картинок станет миллион, и почему следующий модуль занимается **признаками**, а не новыми моделями.

In [ ]:
NEXT_MODULE = ''
assert len(NEXT_MODULE) > 80
print(NEXT_MODULE)